In [2]:
import numpy as np
import h2py
import moments 
import matplotlib.pyplot as plt 
import matplotlib as mpl
import copy
import msprime
import demes 
import demesdraw
import pandas
import pickle
from matplotlib.transforms import ScaledTranslation


mpl.rc("xtick", labelsize=7)
mpl.rc("ytick", labelsize=7)
mpl.rc("font", size=8)
mpl.rc("axes", labelsize=8)
mpl.rc("axes", titlesize=8)
mpl.rc("legend", fontsize=7)
mpl.rc("axes", titlesize="small")

In [64]:
Ne = 1e4
u = 1e-8
rs = [0, 0.5]

def get_divergence_model(s0):
    t0 = 2 * Ne * s0
    builder = demes.Builder()
    builder.add_deme("A", epochs=[{"start_size": Ne, "end_time": t0}])
    builder.add_deme(
        "X", ancestors=["A"], epochs=[{"start_size": Ne, "end_time": 0}])
    builder.add_deme(
        "Y", ancestors=["A"], epochs=[{"start_size": Ne, "end_time": 0}])
    graph = builder.resolve()
    return graph


def get_sigma_h2(yy):
    sig = yy.H2(0, 1, phased=True) / yy.H()[1] ** 2
    return np.array(sig)

def get_exp(s_0):
    ee = [(s_0 ** 2 + 2 * s_0 + 2) / (s_0**2 + 2 * s_0 + 1), 1]
    return np.array(ee)


for s0 in (0.1, 0.2, 0.5):
    yy = moments.Demes.LD(get_divergence_model(s0), sampled_demes=["X", "Y"], u=u, r=rs)
    sig = get_sigma_h2(yy)
    exp = get_exp(s0)
    print(t, sig, exp)

0.5 [1.82628099 0.999835  ] [1.82644628 1.        ]
0.5 [1.69388889 0.9994473 ] [1.69444444 1.        ]
0.5 [1.44222222 0.99778596] [1.44444444 1.        ]


In [65]:
Ne = 1e4
u = 1e-8
rs = [0, 0.5]

def get_introgression_model(p, s0):
    t0 = 2 * Ne * s0
    builder = demes.Builder()
    builder.add_deme("A", epochs=[{"start_size": Ne, "end_time": t0 + 1}])
    builder.add_deme(
        "X", ancestors=["A"], epochs=[{"start_size": Ne, "end_time": 1}])
    builder.add_deme(
        "Y", ancestors=["A"], epochs=[{"start_size": Ne, "end_time": 1}])
    builder.add_deme(
        "Z", ancestors=["X", "Y"], proportions=[p, 1-p], start_time=1, epochs=[{"start_size": 1e4, "end_time": 0}])
    graph = builder.resolve()
    return graph


def get_sigma_h2(yy, idx):
    sig = yy.H2(idx, idx) / yy.H()[0] ** 2
    return np.array(sig)

def get_exp(p_d, s_0):
    ee = [(2 - 2 * p_d  + p_d * (s_0**2 + 2 * s_0 + 2)) / ((1 + p_d * s_0)**2),
          (1 - p_d  + p_d * (s_0**2 + 2 * s_0 + 1)) / ((1 + p_d * s_0)**2)]
    return np.array(ee)

In [66]:
for s0 in (0.1, 0.2, 0.5):
    for p_d in (0.001, 0.1, 0.5):
        yy = moments.Demes.LD(get_introgression_model(p_d, s0), sampled_demes=["Z"], u=u, r=rs)
        sig = get_sigma_h2(yy, 0)
        ee = get_exp(p_d, s0)
        print(t, p_d, sig, ee)

0.5 0.001 [1.99962009 1.00006166] [1.99981002 1.00000999]
0.5 0.1 [1.9663421  1.00080128] [1.98117832 1.00088227]
0.5 0.5 [1.90921522 1.00078486] [1.90929705 1.00226757]
0.5 0.001 [1.99927945 1.00009672] [1.99964006 1.00003994]
0.5 0.1 [1.93708247 1.00296447] [1.96462899 1.00346021]
0.5 0.5 [1.83439603 1.00275926] [1.83471074 1.00826446]
0.5 0.001 [1.9984927  1.00034232] [1.99925025 1.0002495 ]
0.5 0.1 [1.87199407 1.0165568 ] [1.92743764 1.02040816]
0.5 0.5 [1.67842965 1.01326216] [1.68 1.04]


In [70]:
get_exp(0.2, 90)

array([4.59279778, 4.5900277 ])